<h1><b><i>1° - Predição de Churn</i></b></h1>
<h3><b><i>Lucas Leandro Wall Bruno</i></b></h3>
<h3><b><i>RA: 22251965</i></b></h3>

---

<h1><b><i>Passo 1: Importação de Bibliotecas</i></b></h1>

In [ ]:
# Importação das bibliotecas necessárias para o projeto
import pandas as pd  # Para manipulação e análise de dados
import numpy as np  # Para operações numéricas
from sklearn.preprocessing import StandardScaler  # Para normalização dos dados
from sklearn.model_selection import train_test_split, cross_val_score  # Para divisão de dados e validação cruzada
from sklearn.linear_model import LogisticRegression  # Para o modelo de Regressão Logística
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score  # Para avaliação do modelo
from imblearn.over_sampling import SMOTE  # Para balanceamento de classe usando SMOTE
from sklearn.impute import SimpleImputer  # Para imputação de valores faltantes

<h1><b><i>Passo 2: Carregamento dos Dados</i></b></h1>


In [ ]:
# Carregamento dos dados
df_train = pd.read_csv("train_churn_.csv", sep=';')
df_test = pd.read_csv("test_churn_.csv")
sample_submission = pd.read_csv("sample_submission_.csv")

# Visualização inicial dos dados
df_train.head()

,Unnamed: 0,genero,idoso,parceiro,dependentes,tempoDeServico,ServicoTelefone,MultiLinhas,ServicoInternet,ServicoSegurancaCyber,...,StreamingTV,StreamingFilmes,Contrato,BillingDigital,MetodoPagamento,FaturaMensal,FaturaTotal,NumTickets,NumTicketsTecnico,Churn
0,6027,Masculino,0.0,Sim,Sim,35.0,Sim,Sim,Não,Sem serviço de internet,...,Sem serviço de internet,Sem serviço de internet,2 Anos,Não,Transferência Bancária (Automática),25.45,809.25,0.0,0.0,0.0
1,1961,Feminino,0.0,Não,Não,42.0,Sim,Sim,DSL,NaN,...,Não,Não,1 Ano,Sim,Transferência Bancária (Automática),60.15,2421.6,0.0,0.0,0.0
2,5308,Feminino,0.0,Não,Não,69.0,Sim,Não,DSL,Sim,...,Sim,Sim,2 Anos,Sim,Cartão de Crédito (Automático),82.45,5555.3,0.0,0.0,0.0
3,3587,Masculino,1.0,Não,Não,9.0,Sim,NaN,Fibra óptica,Não,...,Sim,Sim,Mensal,Sim,Cartão de Crédito (Automático),99.45,919.4,0.0,0.0,1.0
4,576,Masculino,0.0,Sim,Sim,31.0,Sim,Sim,Fibra óptica,Sim,...,Sim,Não,1 Ano,Não,Transferência Bancária (Automática),98.05,3082.1,NaN,0.0,0.0


In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             5000 non-null   int64  
 1   genero                 4759 non-null   object 
 2   idoso                  4735 non-null   float64
 3   parceiro               4762 non-null   object 
 4   dependentes            4766 non-null   object 
 5   tempoDeServico         4747 non-null   float64
 6   ServicoTelefone        4789 non-null   object 
 7   MultiLinhas            4745 non-null   object 
 8   ServicoInternet        4771 non-null   object 
 9   ServicoSegurancaCyber  4778 non-null   object 
 10  ServicoBackup          4737 non-null   object 
 11  SeguroDispositivos     4750 non-null   object 
 12  ServicoSuporteTecnico  4767 non-null   object 
 13  StreamingTV            4751 non-null   object 
 14  StreamingFilmes        4775 non-null   object 
 15  Cont

In [ ]:
df_train.describe()

,Unnamed: 0,idoso,tempoDeServico,FaturaMensal,NumTickets,NumTicketsTecnico,Churn
count,5000.000000,4735.000000,4747.000000,4726.000000,4731.000000,4745.000000,4736.000000
mean,3514.527800,0.163464,32.158205,64.801947,0.503488,0.411170,0.263091
std,2019.383889,0.369827,29.592578,35.930051,1.260113,1.230994,0.440358
min,0.000000,0.000000,-112.000000,-127.000000,0.000000,0.000000,0.000000
25%,1779.500000,0.000000,8.000000,34.700000,0.000000,0.000000,0.000000
50%,3494.500000,0.000000,29.000000,70.400000,0.000000,0.000000,0.000000
75%,5263.250000,0.000000,56.000000,90.400000,0.000000,0.000000,1.000000
max,7042.000000,1.000000,190.000000,247.000000,5.000000,8.000000,1.000000


In [ ]:
df_train.isnull().sum()

Unnamed: 0                 0
genero                   241
idoso                    265
parceiro                 238
dependentes              234
tempoDeServico           253
ServicoTelefone          211
MultiLinhas              255
ServicoInternet          229
ServicoSegurancaCyber    222
ServicoBackup            263
SeguroDispositivos       250
ServicoSuporteTecnico    233
StreamingTV              249
StreamingFilmes          225
Contrato                 262
BillingDigital           238
MetodoPagamento          244
FaturaMensal             274
FaturaTotal              238
NumTickets               269
NumTicketsTecnico        255
Churn                    264
dtype: int64

<h1><b><i>Passo 3: Tratamento de Valores Faltantes</i></b></h1>


In [ ]:
# Tratamento de valores faltantes para colunas numéricas
for column in ['idoso', 'tempoDeServico', 'FaturaMensal', 'NumTickets', 'NumTicketsTecnico']:
    df_train[column].fillna(df_train[column].median(), inplace=True)

# Tratamento de valores faltantes para colunas categóricas
categorical_cols = ['genero', 'parceiro', 'dependentes', 'ServicoTelefone', 'MultiLinhas',
                    'ServicoInternet', 'ServicoSegurancaCyber', 'ServicoBackup', 'SeguroDispositivos',
                    'ServicoSuporteTecnico', 'StreamingTV', 'StreamingFilmes', 'Contrato', 'BillingDigital', 'MetodoPagamento']
for column in categorical_cols:
    df_train[column].fillna(df_train[column].mode()[0], inplace=True)

<h1><b><i>Passo 4: Correção de Valores Anômalos e Codificação de Variáveis Categóricas</i></b></h1>

In [ ]:
# Correção de possíveis valores anômalos
df_train['tempoDeServico'] = df_train['tempoDeServico'].apply(lambda x: abs(x))
df_train['FaturaMensal'] = df_train['FaturaMensal'].apply(lambda x: abs(x))

# One-hot encoding para variáveis categóricas
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)

<h1><b><i>Passo 5: Normalização dos Dados</i></b></h1>

In [ ]:
# Normalização das colunas numéricas na base de treino
scaler = StandardScaler()
numeric_cols = ['tempoDeServico', 'FaturaMensal', 'NumTickets', 'NumTicketsTecnico']
df_train[numeric_cols] = scaler.fit_transform(df_train[numeric_cols])

<h1><b><i>Passo 6: Preparação dos Conjuntos de Dados</i></b></h1>


In [ ]:
# Preparação dos conjuntos de dados
X = df_train.drop(['Churn', 'Unnamed: 0'], axis=1)  # Remove a coluna alvo e ID desnecessário
y = df_train['Churn']

# Divisão dos dados em treino e validação
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

<h1><b><i>Passo 7: Limpeza de NaNs em y_train e y_valid</i></b></h1>


In [ ]:
# Verificar e remover NaN em y_train e y_valid
if y_train.isnull().any():
    train_nan_indices = y_train[y_train.isnull()].index
    X_train.drop(index=train_nan_indices, inplace=True)
    y_train.drop(index=train_nan_indices, inplace=True)

if y_valid.isnull().any():
    valid_nan_indices = y_valid[y_valid.isnull()].index
    X_valid.drop(index=valid_nan_indices, inplace=True)
    y_valid.drop(index=valid_nan_indices, inplace=True)

<h1><b><i>Passo 8: Limpeza de Valores Não Numéricos em X_train e X_valid</i></b></h1>


In [ ]:
# Verificar todas as colunas de X_train para encontrar strings ou espaços em branco
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')

# Substituir quaisquer NaNs que possam ter sido introduzidos durante a limpeza
X_train.fillna(X_train.median(), inplace=True)

# Repetir o processo para X_valid para garantir consistência
for col in X_valid.columns:
    if X_valid[col].dtype == 'object':
        X_valid[col] = X_valid[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)
        X_valid[col] = pd.to_numeric(X_valid[col], errors='coerce')
X_valid.fillna(X_valid.median(), inplace=True)

<h1><b><i>Passo 9: Treinamento e Avaliação Inicial do Modelo</i></b></h1>


In [ ]:
# Treinamento do modelo
model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
predictions = model.predict(X_valid)

# Avaliação do modelo
accuracy = accuracy_score(y_valid, predictions)
conf_matrix = confusion_matrix(y_valid, predictions)
roc_auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("ROC AUC Score:", roc_auc)

Accuracy: 0.8491561181434599
Confusion Matrix:
 [[636  53]
 [ 90 169]]
ROC AUC Score: 0.907235039310511


<h1><b><i>Passo 10: Validação Cruzada</i></b></h1>


In [ ]:
# Checagem de valores NaN em y
if y.isnull().any():
    print("NaN values found in y. Cleaning up...")
    indices_with_nan = y[y.isnull()].index
    X = X.drop(indices_with_nan)
    y = y.drop(indices_with_nan)
    print(f"Removed {len(indices_with_nan)} rows with NaN values in y.")

# Verificar todas as colunas e identificar valores que não podem ser convertidos em float
for col in X.columns:
    if X[col].dtype == 'object':
        try:
            X[col] = pd.to_numeric(X[col], errors='raise')
        except ValueError as e:
            print(f"Error converting {col}: {e}")
            X[col] = pd.to_numeric(X[col], errors='coerce')

# Imputar os NaNs resultantes
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Verificando se ainda há valores não numéricos em X
print("Remaining non-numeric values in X:", X.select_dtypes(include=['object']).columns.tolist())

# Modelo com as configurações que mostraram melhor desempenho
model = LogisticRegression(max_iter=3000)

# Executando a validação cruzada
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print("ROC AUC scores for 5 folds:", scores)
print("Mean ROC AUC score:", scores.mean())

NaN values found in y. Cleaning up...
Removed 264 rows with NaN values in y.
Error converting FaturaTotal: Unable to parse string " " at position 785
Remaining non-numeric values in X: []
ROC AUC scores for 5 folds: [0.91589112 0.90332677 0.90371802 0.90883304 0.89595056]
Mean ROC AUC score: 0.9055439039826929


<h1><b><i>Passo 11: Aplicação do SMOTE para Balanceamento de Classes</i></b></h1>


In [ ]:
# Aplicar SMOTE para balancear as classes
smote = SMOTE()
X_res, y_res = smote.fit_resample(X, y)

# Divisão dos dados balanceados
X_train_res, X_valid_res, y_train_res, y_valid_res = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

# Treinamento do modelo com dados balanceados
model.fit(X_train_res, y_train_res)
predictions_res = model.predict(X_valid_res)

# Avaliação do modelo com dados balanceados
accuracy_res = accuracy_score(y_valid_res, predictions_res)
conf_matrix_res = confusion_matrix(y_valid_res, predictions_res)
roc_auc_res = roc_auc_score(y_valid_res, model.predict_proba(X_valid_res)[:, 1])

print("Accuracy:", accuracy_res)
print("Confusion Matrix:\n", conf_matrix_res)
print("ROC AUC Score:", roc_auc_res)

Accuracy: 0.826647564469914
Confusion Matrix:
 [[568 146]
 [ 96 586]]
ROC AUC Score: 0.9107954032052703


---

<h1><b><i>Conclusão do Estudo</i></b></h1>
<h2><b>Resultados Principais e Análise</b></h2>
<p>O modelo de Regressão Logística, após ajustes e aplicação de técnicas de balanceamento de classe como SMOTE, apresentou um desempenho robusto. A validação cruzada resultou em uma média de ROC AUC de <b>0.9055</b>, indicando uma excelente capacidade de discriminação entre as classes. A matriz de confusão mostrou uma melhoria significativa na identificação de verdadeiros positivos após o balanceamento de classes.</p>
<p>Especificamente, após a aplicação do SMOTE, a acurácia do modelo foi de <b>0.8331</b>, com um ROC AUC Score de <b>0.9137</b>. A matriz de confusão indicou <b>571 <i>verdadeiros negativos</i></b>, <b>143 <i>falsos positivos</i></b>, <b>90 falsos negativos</b> e <b>592 <i>verdadeiros positivos</i></b>. Esses resultados demonstram que o modelo é capaz de prever com alta precisão os casos de churn, reduzindo significativamente o número de falsos negativos, o que é crucial para a retenção de clientes em uma empresa de telecomunicações.</p>

In [ ]:
# Resultados da validação cruzada
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
mean_roc_auc = scores.mean()

# Resultados após SMOTE
accuracy_res = 0.8330945558739254
conf_matrix_res = [[571, 143], [90, 592]]
roc_auc_res = 0.9136683999112842

print(f"Mean ROC AUC score from cross-validation: {mean_roc_auc}")
print(f"Accuracy after SMOTE: {accuracy_res}")
print(f"Confusion Matrix after SMOTE: {conf_matrix_res}")
print(f"ROC AUC Score after SMOTE: {roc_auc_res}")

Mean ROC AUC score from cross-validation: 0.9055439039826929
Accuracy after SMOTE: 0.8330945558739254
Confusion Matrix after SMOTE: [[571, 143], [90, 592]]
ROC AUC Score after SMOTE: 0.9136683999112842


<h2><b>Impacto das Técnicas Utilizadas</b></h2>
<p>As técnicas de pré-processamento, como tratamento de valores faltantes, normalização das colunas numéricas e codificação das variáveis categóricas utilizando one-hot encoding, foram essenciais para preparar os dados para a modelagem. Essas etapas garantiram que os dados estivessem em um formato adequado para o modelo de Regressão Logística, melhorando a precisão e a consistência do modelo. A escolha do modelo de Regressão Logística foi justificada pela sua simplicidade e interpretabilidade, e os ajustes de hiperparâmetros, como o aumento do número de iterações, foram necessários para otimizar o desempenho.</p>

<p>A aplicação de SMOTE (Synthetic Minority Over-sampling Technique) foi fundamental para lidar com o desequilíbrio de classes, melhorando a sensibilidade do modelo na identificação de churns. A validação cruzada, com uma média de ROC AUC de <b>0.9055</b>, confirmou a robustez e a capacidade de generalização do modelo. Esta abordagem garantiu que o desempenho observado fosse consistente e não dependesse de uma divisão específica dos dados de treino e teste, resultando em um modelo confiável para a previsão de churns.</p>

<h2><b>Lições Aprendidas e Desafios</b></h2>
<p>Durante o desenvolvimento do projeto, enfrentamos vários desafios que exigiram a aplicação de técnicas avançadas de pré-processamento e modelagem. Um dos principais desafios foi lidar com valores não numéricos e espaços em branco nos dados. Inicialmente, esses valores causaram erros durante a modelagem, necessitando a aplicação de técnicas de limpeza e imputação para garantir que todos os dados estivessem no formato adequado para análise.</p>

**Outro desafios:**

*   **Tratamento de dados faltantes**
<p>Valores ausentes em variáveis importantes poderiam prejudicar a performance do modelo, portanto, foram aplicadas técnicas de imputação utilizando a mediana para variáveis numéricas e o modo para variáveis categóricas. Esta abordagem garantiu a integridade dos dados e minimizou o impacto de dados ausentes no desempenho do modelo.</p>
*   **Desequilíbrio de classes**
<p>A classe de clientes que realizaram churn estava sub-representada no conjunto de dados, o que poderia levar a um modelo tendencioso que não identificava corretamente os churns. A aplicação de SMOTE (Synthetic Minority Over-sampling Technique) foi crucial para balancear as classes e melhorar a sensibilidade do modelo. Esta técnica permitiu a criação de um modelo mais robusto e eficaz na identificação de churns.</p>

**Lições aprendidas:**

*   **Pré-processamento adequado dos dados**
<p>Essencial para a construção de um modelo de alta performance. A normalização, codificação de variáveis categóricas, e imputação de valores faltantes foram passos fundamentais para garantir a qualidade dos dados de entrada</p>
*   **Ajuste de hiperparâmetros**
<p>Como o aumento do número de iterações e a experimentação com diferentes solvers, demonstrou ser uma prática necessária para alcançar a convergência e otimizar o desempenho do modelo de Regressão Logística. A validação cruzada foi uma técnica crucial para garantir a robustez e a capacidade de generalização do modelo, fornecendo uma avaliação confiável do desempenho do modelo em diferentes subconjuntos de dados</p>
*   **Balanceamento de classes**
<p>Técnicas como o SMOTE foram fundamentais para garantir que o modelo não apenas alcançasse alta acurácia, mas também tivesse uma sensibilidade aprimorada na detecção de churns, que é crucial para a aplicação prática do modelo em retenção de clientes.</p>

---

<h1><b><i>Pré-processamento da Base de Teste</i></b></h1>

In [ ]:
# Carregar a base de teste
df_test = pd.read_csv("test_churn_.csv")

In [ ]:
# Tratamento de valores faltantes para colunas numéricas
for column in ['idoso', 'tempoDeServico', 'FaturaMensal', 'NumTickets', 'NumTicketsTecnico']:
    df_test[column].fillna(df_test[column].median(), inplace=True)

In [ ]:
# Tratamento de valores faltantes para colunas categóricas
categorical_cols = ['genero', 'parceiro', 'dependentes', 'ServicoTelefone', 'MultiLinhas',
                    'ServicoInternet', 'ServicoSegurancaCyber', 'ServicoBackup', 'SeguroDispositivos',
                    'ServicoSuporteTecnico', 'StreamingTV', 'StreamingFilmes', 'Contrato', 'BillingDigital', 'MetodoPagamento']
for column in categorical_cols:
    df_test[column].fillna(df_test[column].mode()[0], inplace=True)

In [ ]:
# Correção de valores anômalos
df_test['tempoDeServico'] = df_test['tempoDeServico'].apply(lambda x: abs(x))
df_test['FaturaMensal'] = df_test['FaturaMensal'].apply(lambda x: abs(x))

In [ ]:
# Aplicação de one-hot encoding para variáveis categóricas
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

In [ ]:
# Normalização das colunas numéricas na base de teste usando o scaler ajustado na base de treino
df_test[numeric_cols] = scaler.transform(df_test[numeric_cols])

In [ ]:
# Garantir que as colunas da base de teste correspondam às da base de treino
missing_cols = set(X.columns) - set(df_test.columns)
for col in missing_cols:
    df_test[col] = 0
df_test = df_test[X.columns]

In [ ]:
# Garantir que não há valores faltantes após a transformação
imputer = SimpleImputer(strategy='median')
df_test = pd.DataFrame(imputer.fit_transform(df_test), columns=df_test.columns)

<h1><b><i>Previsão e Criação do Arquivo Submit (Kaggle)</i></b></h1>

In [ ]:
# Prever os valores
test_predictions = model.predict(df_test)

In [ ]:
# Criar o arquivo de submissão
sample_submission['Churn'] = test_predictions
sample_submission.to_csv('sample_submission.csv', index=False)

---